# MALDI lipid data — summary analysis

A focused look at the raw lipid dataset in three parts:

1. **General statistics** — value distributions, coverage, dynamic range.
2. **Lipid–lipid correlogram** — how strongly lipids co-vary, and effective dimensionality.
3. **Region-identity effect** — how much of each lipid's value is explained by the Allen atlas region.


In [ ]:
import numpy as np, pandas as pd
import matplotlib, matplotlib.pyplot as plt

DATA   = '/home/casap/mlibra/mlibra_data'
MALDI  = f'{DATA}/maindata_minimal.parquet'
TEMPLATE = f'{DATA}/reference_image.npy'
ANNOT    = f'{DATA}/level_15annot.npy'      # Allen atlas labels (0=bg, 997=root)

MAX_VOX  = 200_000     # subsample for correlation / region ANOVA (stable)
rng = np.random.default_rng(0)


In [ ]:
# --- load + tissue mask + region labels -------------------------------------
df = pd.read_parquet(MALDI)
meta = ['x','y','Sample','Section','xccf','yccf','zccf',
        'x_index','y_index','z_index','SampleSection']
lipids = [c for c in df.columns if c not in meta]
print(f'{len(df):,} points  |  {len(lipids)} lipids  |  {df.Sample.nunique()} mice')

template = np.load(TEMPLATE); annot = np.load(ANNOT)
z,y,x = (df[c].values.astype(int) for c in ['z_index','y_index','x_index'])
ib = ((z>=0)&(z<template.shape[0])&(y>=0)&(y<template.shape[1])&(x>=0)&(x<template.shape[2]))
tissue = np.zeros(len(df), bool); region = np.zeros(len(df), np.int64)
tissue[ib] = template[z[ib],y[ib],x[ib]] > 5
region[ib] = annot[z[ib], y[ib], x[ib]]
df['tissue'] = tissue; df['region'] = region

dft = df[df.tissue].copy()
print(f'{len(dft):,} tissue voxels  |  {dft.region.nunique()} region labels')

# log1p lipid values (clip negatives), for all distribution/correlation work
Ylog_full = np.log1p(np.clip(dft[lipids].values.astype(np.float32), 0, None))
# subsampled working set (shared by correlogram + region ANOVA)
idx = rng.choice(len(dft), min(MAX_VOX, len(dft)), replace=False)
Ylog = Ylog_full[idx]
reg  = dft['region'].values[idx]


## 1. General statistics & value distributions

In [ ]:
# Per-lipid summary: coverage (frac > 0), abundance, dynamic range (in log1p space)
raw = np.clip(dft[lipids].values.astype(np.float32), 0, None)
coverage = (raw > 0).mean(0)                       # fraction of voxels detected
summary = pd.DataFrame({
    'coverage'    : coverage,
    'mean_log1p'  : Ylog_full.mean(0),
    'median_log1p': np.median(Ylog_full, 0),
    'p99_log1p'   : np.percentile(Ylog_full, 99, 0),
}, index=lipids).sort_values('mean_log1p', ascending=False)
print('=== dataset overview ===')
print(f'lipids: {len(lipids)}   tissue voxels: {len(dft):,}')
print(f'median per-lipid coverage: {np.median(coverage):.1%}')
print(f'lipids with <50% coverage: {(coverage<0.5).sum()}')
summary.head(15)


In [ ]:
# Value distributions
fig, ax = plt.subplots(1, 2, figsize=(13,4))
ax[0].hist(Ylog[Ylog>0].ravel(), bins=120, color='steelblue')
ax[0].set(title='All lipid values (log1p, detected only)', xlabel='log1p intensity', ylabel='count')
ax[1].hist(coverage, bins=30, color='indianred')
ax[1].set(title='Per-lipid coverage', xlabel='fraction of voxels detected', ylabel='# lipids')
plt.tight_layout(); plt.show()

# small multiples: 12 lipids spanning the abundance range
pick = summary.index[np.linspace(0, len(summary)-1, 12).astype(int)]
fig, axes = plt.subplots(3, 4, figsize=(15,8))
for a, name in zip(axes.ravel(), pick):
    v = Ylog_full[:, lipids.index(name)]
    a.hist(v[v>0], bins=60, color='slategray')
    a.set_title(name, fontsize=9); a.tick_params(labelsize=7)
fig.suptitle('Value distributions across the abundance range (log1p, detected)', y=1.01)
plt.tight_layout(); plt.show()


## 2. Lipid–lipid correlogram

How strongly lipids co-vary, and the effective dimensionality of the panel.

Lipids are ordered by their loading on the **first eigenvector of the correlation
matrix**. PC1 is by far the dominant axis of co-variation, so this ordering places
every lipid on a single interpretable axis and resolves the panel into two opposed
groups (the "poles").


In [ ]:
C = np.corrcoef(Ylog, rowvar=False)                # (n_lipids, n_lipids)

# --- eigendecomposition: PC1 is the dominant axis of co-variation ------------
evals, evecs = np.linalg.eigh(C)                   # ascending
w   = evals[::-1].clip(min=0)                      # descending eigenvalues
pc1 = evecs[:, -1]
if pc1.mean() < 0:
    pc1 = -pc1                                     # eigenvector sign is arbitrary
order = np.argsort(pc1)                            # <-- sort by 1st eigenvector
pole  = pc1 >= 0                                   # the two ends of the PC1 axis

POS, NEG = '#b2182b', '#2166ac'                    # ends of the RdBu_r ramp
pole_cmap = matplotlib.colors.ListedColormap([NEG, POS])

fig, ax = plt.subplots(figsize=(7.6, 7))
im = ax.imshow(C[np.ix_(order, order)], cmap='RdBu_r', vmin=-1, vmax=1,
               interpolation='nearest')
ax.set(title=f'Lipid\u2013lipid correlation, sorted by 1st eigenvector'
             f'  (PC1 = {w[0]/len(lipids):.1%} of variance)',
       xticks=[], yticks=[])

# pole membership strip above the matrix, in the same order as the rows
strip = ax.inset_axes([0, 1.012, 1, 0.022])
strip.imshow(pole[order][None, :], aspect='auto', cmap=pole_cmap, vmin=0, vmax=1)
strip.set_axis_off()

cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
cb.set_label('Pearson r')
cb.outline.set_visible(False)

handles = [plt.matplotlib.patches.Patch(facecolor=POS,
               label=f'PC1 + pole  ({pole.sum()} lipids)'),
           plt.matplotlib.patches.Patch(facecolor=NEG,
               label=f'PC1 \u2212 pole  ({(~pole).sum()} lipids)')]
ax.legend(handles=handles, loc='upper center', ncol=2, frameon=False,
          bbox_to_anchor=(0.5, -0.02))
plt.show()

off = C[np.triu_indices_from(C, 1)]
pr  = (w.sum()**2)/(w**2).sum()                    # participation ratio = effective rank
cum = np.cumsum(w)/w.sum()
print(f'mean |r| (off-diag):        {np.abs(off).mean():.3f}')
print(f'frac pairs |r| > 0.5:       {(np.abs(off)>0.5).mean():.1%}')
print(f'frac pairs r < 0:           {(off<0).mean():.1%}')
print(f'PC1 share of variance:      {w[0]/len(lipids):.1%}')
print(f'effective rank (PR):        {pr:.1f}  of {len(lipids)}')
print(f'components for 90% var:     {int(np.searchsorted(cum,0.90))+1}')
print(f'components for 95% var:     {int(np.searchsorted(cum,0.95))+1}')
print(f'PC1 poles:                  {pole.sum()} / {(~pole).sum()} lipids')

## 3. Region-identity effect

For each lipid, the fraction of its variance explained by the Allen atlas region ($\eta^2$ from a one-way ANOVA over region labels). This is the *coarse-atlas* ceiling — how much a lipid's value is simply "which region am I in."

`EXCLUDE_ROOT` drops the 997 catch-all (≈57% of voxels, anatomically heterogeneous); toggle to see its effect.

In [ ]:
EXCLUDE_ROOT = True

keep = reg > 0                                     # drop background (0)
if EXCLUDE_ROOT: keep &= (reg != 997)
Yk, rk = Ylog[keep], reg[keep]

# vectorized one-way ANOVA eta^2 per lipid
labs, inv = np.unique(rk, return_inverse=True)
n_g   = np.bincount(inv)                                        # (n_regions,)
sum_g = np.zeros((len(labs), Yk.shape[1]), np.float64)
np.add.at(sum_g, inv, Yk)
mu_g  = sum_g / n_g[:,None]                                     # region means
mu    = Yk.mean(0)
ss_between = (n_g[:,None] * (mu_g - mu)**2).sum(0)
ss_total   = ((Yk - mu)**2).sum(0)
eta2 = ss_between / np.where(ss_total>0, ss_total, np.nan)
eta2 = pd.Series(eta2, index=lipids).sort_values(ascending=False)

print(f'regions used: {len(labs)}   voxels: {len(Yk):,}   (EXCLUDE_ROOT={EXCLUDE_ROOT})')
print(f'mean  region-explained variance: {eta2.mean():.1%}')
print(f'median region-explained variance: {eta2.median():.1%}')


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14,5))
ax[0].hist(eta2.values, bins=30, color='seagreen')
ax[0].axvline(eta2.mean(), color='k', ls='--', label=f'mean {eta2.mean():.1%}')
ax[0].set(title='Region-explained variance per lipid ($\\eta^2$)',
          xlabel='fraction of variance from region', ylabel='# lipids'); ax[0].legend()
top = eta2.head(15)[::-1]
ax[1].barh(range(len(top)), top.values, color='seagreen')
ax[1].set_yticks(range(len(top))); ax[1].set_yticklabels(top.index, fontsize=8)
ax[1].set(title='Most region-determined lipids', xlabel='$\\eta^2$')
plt.tight_layout(); plt.show()
